In [2]:
%load_ext autoreload

%autoreload 2

In [3]:
import os 
import sys

import pandas as pd
import numpy as np

import pickle

from preproces_prod3 import *
from matching_case_control import *

import inv
from IPython.core.display import display

import pandas as pd
from datetime import datetime, timedelta
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import seaborn as sns   
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
import os
from pathlib import Path
import seaborn as sns
import numpy as np
from lifelines.utils import to_long_format
from lifelines.utils import add_covariate_to_timeline
from lifelines import CoxTimeVaryingFitter
from statsmodels.distributions.empirical_distribution import ECDF
from scipy.stats import ks_2samp
from scipy import stats
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as mtick
from lifelines import KaplanMeierFitter
from lifelines.statistics import pairwise_logrank_test
from lifelines.plotting import add_at_risk_counts
import warnings
from preproces_prod4_2025_update import *
from scipy.stats import norm
import gspread
from oauth2client.service_account import ServiceAccountCredentials
warnings.filterwarnings("ignore")

display.max_output_lines = 500  # Adjust the number as needed
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

In [3]:
df_vrs_match_case, _ = call_data_2025_update('COHORTE_NIRSE_ACTUALIZADA_04_12_2025_ENCR.csv',
                                             fecha_referencia_nacido_str = '2024-10-01',
                                             fecha_referencia_campaña_str = '2025-03-01',
                                             cohort='2025',
                                             eliminar_inmunes_pre_season=True, 
                                             T_inicial = pd.to_datetime('2025-03-01'), 
                                             fecha_dt = pd.to_datetime('2025-08-31'))

n_rows_inicial= 334025
['DESCONOCIDO---->None' 'Región De Antofagasta---->ANTOFAGASTA'
 'Región De Arica Parinacota---->ARICA Y PARINACOTA'
 'Región De Atacama---->ATACAMA'
 'Región De Aysén del General Carlos Ibañez del Campo---->AISEN'
 'Región De Coquimbo---->COQUIMBO' 'Región De La Araucanía---->ARAUCANIA'
 'Región De Los Lagos---->LOS LAGOS' 'Región De Los Ríos---->LOS RIOS'
 'Región De Magallanes y de la Antártica Chilena---->MAGALLANES Y ANTARTICA'
 'Región De Tarapacá---->TARAPACA' 'Región De Valparaíso---->VALPARAISO'
 'Región De Ñuble---->NUBLE' 'Región Del Bíobío---->BIOBIO'
 "Región Del Libertador Gral. B. O'Higgins---->O'HIGGINS"
 'Región Del Maule---->MAULE'
 'Región Metropolitana de Santiago---->METROPOLITANA']
n_rows_post_prefiltred= 334025
Datos perdidos por muertes:  962
ruts perdidos por filtro semanas y peso:  540
Droped intersex: 5
Datos perdidos por edad madre atípica: 247
Datos perdidos por fecha ingreso menor a fecha nacimiento: 12
vrs en los primeros 7 dias de 

In [4]:
icd10_codes_fr = [
    "N390",   # Infección del tracto urinario (sin síntomas respiratorios) INFECCION DE VÍAS URINARIAS, SITIO NO ESPECIFICADO
    "A090",     # Gastroenteritis aguda (sin síntomas respiratorios) OTRAS GASTROENTERITIS Y COLITIS NO ESPECIFICADAS DE ORIGEN INFECCIOSO
    "A099",     # Gastroenteritis aguda (sin síntomas respiratorios) GASTROENTERITIS Y COLITIS DE ORIGEN NO ESPECIFICADO
    "A09X",     # Gastroenteritis aguda (sin síntomas respiratorios) DIARREA Y GASTROENTERITIS DE PRESUNO ORIGEN INFECCIOSO
    "R100",  # Cólico infantil (sin fiebre ni síntomas respiratorios) ABDOMEN AGUDO
    "R101",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR ABDOMINAL LOCALIZADO EN PARTE SUPERIOR
    "R102",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR PELVICO Y PERINEAL
    "R103",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR LOCALIZADO EN OTRAS PARTES INFERIORES DEL ABDOMEN
    "R104",  # Cólico infantil (sin fiebre ni síntomas respiratorios) OTROS DOLORES ABDOMINALES Y LOS NO ESPECIFICADOS
    "R11X",  # Cólico infantil (sin fiebre ni síntomas respiratorios) NAUSEA Y VOMITO
    "R634",   # Pérdida de peso (sin fiebre ni síntomas respiratorios) PERDIDA ANORMAL DE PESO
    "R633",   # Dificultades de alimentación (sin fiebre ni síntomas respiratorios) DIFICULTADES Y MALA ADMINISTRACION DE LA ALIMENTACION
    "P599",   # Ictericia neonatal (sin fiebre ni síntomas respiratorios) ICTERICIA NEONATAL, NO ESPECIFICADA
    "R681",  # Llanto anormal (sin fiebre ni síntomas respiratorios) SINTOMAS NO ESPECIFICOS PROPIOS DE LA INFANCIA
    "S099",  # Traumatismo craneal (sin fiebre ni síntomas respiratorios)
    "Z539"    # Cirugía de emergencia (sin fiebre ni síntomas respiratorios)
    ################### AÑADIDOS POR MI #######################
    'A099', 
    'A080', 
    'A083', 
    'A082', 
    'A084',
    'A090', 
    'A081', 
    'A085'
]

list_experiments=[]

df = (df_vrs_match_case
      .drop(columns=[col for col in df_vrs_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) | 
                                               (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                                               (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                                               (col.endswith('Dall')) )])
      .assign(fecha_nac = lambda x: pd.to_datetime(x.fecha_nac, infer_datetime_format=True),
              year_nac = lambda x: x.fecha_nac.dt.year,
              mes_nac_name = lambda x: x.fecha_nac.dt.month_name(),  
              semana_nac = lambda x: x.fecha_nac.dt.isocalendar().week.astype(int),
              criterio_pali_normal = lambda x: ((x.SEMANAS<32) | (x.PESO<1500)).astype(int),
            #   pali = lambda x: np.where(x.criterio_pali_normal==1, 1, 
            #                             np.where((x.year_nac==2023) & (x.month_nac.between(7, 9, inclusive='both')) & (x.SEMANAS<=34) & ((x.PESO<2500)), 1, 0))
            )
      .copy()
      )


# df.loc[df.RUN == 'ac483764636448d753930868dd3192a785e3728a2d81b3fc44dba45c3506255a', 'region'] = 'METROPOLITANA'
# df.loc[df.COMUNA == 12202, 'region'] = 'MAGALLANES Y ANTARTICA'

########################################################################################################################
match_vars_distance_nn=['edad_relativa','ingreso_relativo']
match_vars_exact_nn = ['eleg_2025','region','PREVI','prematuro'] #NOMBRE_REGION
weights = {'edad_relativa': 1,'ingreso_relativo': 2} 

match_vars_distance_IP = ['edad_relativa']
match_vars_distance_IP_francia = ['edad_relativa','ingreso_relativo']

match_vars_exact_IP_previ = ['prematuro','region','eleg_2025','PREVI'] # region
match_vars_exact_IP_not_previ = ['prematuro','region','eleg_2025','mes_nac_name','semana_nac']

########################################################## VRS ##########################################################

filtro_comba_dict = {'VRS_general': 'MARCA==0'}

def ratio_unif(ratio,dic_filtros=filtro_comba_dict):
    dic_ratio_unif = { }

    for key in dic_filtros.keys():
        dic_ratio_unif[key] = ratio
    return dic_ratio_unif

dic_ratio_unif = ratio_unif(ratio='1:7')

df_copy = df.copy()
df_francia = df.query('(fechaIng_any.notna()) & (diag_1_leter!="J")').copy()


list_experiments_all_born = results_match(df_case_study=df_copy,
                                 df_control_study=df_copy.sample(frac=0.25, random_state=42), #.sample(frac=0.25, random_state=42)
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ, # ['prematuro','region','group','cardio1']
                                 weights={'edad_relativa': 1}, 
                                 list_experiments = [],
                                 nn=False,
                                 ratio="1:1",
                                 covs = ['sexo','SEMANAS','PESO'], #cardio1   ['sexo','PESO','SEMANAS'] ,'cardio1','prematuro'
                                 dic_ratio = dic_ratio_unif) # ['sexo','SEMANAS','PESO','cardio1','prematuro']

# with open("list_experiments_all_born.pkl", "wb") as f:
#     pickle.dump(list_experiments_all_born, f)

# df_copy = df.copy()
# #df_francia = df.query('(fechaIng_any.notna()) & (diag_1_leter!="J")').copy()
# df_francia = df.query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))').copy()

# list_experiments_francia_not_previ = results_match(df_case_study=df_copy,
#                                  df_control_study=df_francia,
#                                  filtros_dic=filtro_comba_dict,
#                                  match_vars_distance_nn=match_vars_distance_nn,
#                                  match_vars_exact_nn=match_vars_exact_nn,
#                                  match_vars_distance_IP=match_vars_distance_IP_francia,
#                                  match_vars_exact_IP= ['prematuro','Macrozones','group','PREVI'], #match_vars_exact_IP_not_previ, cardio1
#                                  weights=weights,
#                                  list_experiments = [],
#                                  nn=False,
#                                  ratio="1:1",
#                                  covs =  ['sexo','SEMANAS','PESO'])  #cardio1 



# with open("list_experiments_francia_not_previ.pkl", "wb") as f:
#     pickle.dump(list_experiments_francia_not_previ, f)

#df_final = summary_eff_aux(list_experiments_francia_not_previ = list_experiments_francia_not_previ)
df_final = summary_eff_aux(list_experiments_all_born=list_experiments_all_born) #list_experiments_all_born, list_experiments_francia_not_previ
df_final

VRS_general 1251 28473 ['sexo', 'SEMANAS', 'PESO']
creacion conjuntos A_i time: 66.49192214012146
Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-07
creacion variables time: 70.00050139427185
0.07034498347128311
optimize model 1 time: 2.190995693206787
optimize model 2 time: 109.78534054756165
matched_data time: 4.601409912109375
Total cases = 1251, Total controls = 28473
Total cases matched is : 1125, Total control matched is : 7875
ratio: 1:7

Odds Ratios y Efectividad:
                      Coeficientes       OR  Efectividad      0.975      0.025
estado_inmunizacion     -1.889945  0.15108    84.891984  88.286673  80.513466

IC disntace: 0.5089963860115905


all_born                               
                      efectividad cases matched controls_matched
filtro                                                          
VRS_general  84.89 (80.51; 88.29)  1251    1125             7875

In [4]:
df_vrs_match_case_24, _ = (call_data_2025_update('COHORTE_NIRSE_ACTUALIZADA_04_12_2025_ENCR.csv',
                                             fecha_referencia_nacido_str = '2023-10-01',
                                             fecha_referencia_campaña_str = '2024-03-01',
                                             cohort='2024',
                                             eliminar_inmunes_pre_season=True, 
                                             T_inicial = pd.to_datetime('2024-04-01'), 
                                             fecha_dt = pd.to_datetime('2024-09-30')))

                        #    .drop(columns=[
                        #        col for col in df_vrs_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) |
                        #                                                     (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                        #                                                     (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                        #                                                     (col.endswith('Dall')) 
                        #                                                     )
                        #                   ]
                        #          )

n_rows_inicial= 334025
['DESCONOCIDO---->None' 'Región De Antofagasta---->ANTOFAGASTA'
 'Región De Arica Parinacota---->ARICA Y PARINACOTA'
 'Región De Atacama---->ATACAMA'
 'Región De Aysén del General Carlos Ibañez del Campo---->AISEN'
 'Región De Coquimbo---->COQUIMBO' 'Región De La Araucanía---->ARAUCANIA'
 'Región De Los Lagos---->LOS LAGOS' 'Región De Los Ríos---->LOS RIOS'
 'Región De Magallanes y de la Antártica Chilena---->MAGALLANES Y ANTARTICA'
 'Región De Tarapacá---->TARAPACA' 'Región De Valparaíso---->VALPARAISO'
 'Región De Ñuble---->NUBLE' 'Región Del Bíobío---->BIOBIO'
 "Región Del Libertador Gral. B. O'Higgins---->O'HIGGINS"
 'Región Del Maule---->MAULE'
 'Región Metropolitana de Santiago---->METROPOLITANA']
n_rows_post_prefiltred= 334025
Datos perdidos por muertes:  1490
ruts perdidos por filtro semanas y peso:  519
Droped intersex: 22
Datos perdidos por edad madre atípica: 210
Datos perdidos por fecha ingreso menor a fecha nacimiento: 19
vrs en los primeros 7 dias d

In [18]:
df_vrs_match_case_24 = (df_vrs_match_case_24
      .drop(columns=[col for col in df_vrs_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) | 
                                               (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                                               (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                                               (col.endswith('Dall')) )])
      )

In [ ]:
list_experiments=[]

df = df_vrs_match_case_24.copy()


match_vars_distance_IP = ['edad_relativa']
match_vars_distance_IP_francia = ['edad_relativa','ingreso_relativo']

match_vars_exact_IP_previ = ['prematuro','region','eleg_2025','PREVI','mes_nac_name','semana_nac'] # region
match_vars_exact_IP_not_previ = ['prematuro','region','eleg_2025']

########################################################## VRS ##########################################################

filtro_comba_dict = {'VRS_general': 'MARCA==0'}

def ratio_unif(ratio,dic_filtros=filtro_comba_dict):
    dic_ratio_unif = { }

    for key in dic_filtros.keys():
        dic_ratio_unif[key] = ratio
    return dic_ratio_unif

dic_ratio_unif = ratio_unif(ratio='1:7')

df_copy = df.copy()

match_vars_distance_nn=['edad_relativa','ingreso_relativo']
match_vars_exact_nn = ['eleg_2025','region','PREVI','prematuro'] #NOMBRE_REGION
weights = {'edad_relativa': 1,'ingreso_relativo': 2} 

match_vars_distance_IP = ['edad_relativa']
match_vars_distance_IP_francia = ['edad_relativa','ingreso_relativo']

match_vars_exact_IP_previ = ['prematuro','region','eleg_2025','PREVI'] # region
match_vars_exact_IP_not_previ = ['prematuro','region','eleg_2025','mes_nac_name','semana_nac']



list_experiments_all_born = results_match(df_case_study=df_copy,
                                 df_control_study=df_copy.sample(frac=0.25, random_state=42), #.sample(frac=0.25, random_state=42)
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ, # ['prematuro','region','group','cardio1']
                                 weights={'edad_relativa': 1}, 
                                 list_experiments = [],
                                 nn=False,
                                 ratio="1:1",
                                 covs = ['sexo','SEMANAS','PESO'], #cardio1   ['sexo','PESO','SEMANAS'] ,'cardio1','prematuro'
                                 dic_ratio = dic_ratio_unif) # ['sexo','SEMANAS','PESO','cardio1','prematuro']

df_final = summary_eff_aux(list_experiments_all_born=list_experiments_all_born) #list_experiments_all_born, list_experiments_francia_not_previ
df_final

VRS_general 1299 38012 ['sexo', 'SEMANAS', 'PESO']
creacion conjuntos A_i time: 111.2628219127655
Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-07
creacion variables time: 115.21620297431946
0.05772250926237253
optimize model 1 time: 2.1564137935638428
optimize model 2 time: 55.354724645614624
matched_data time: 4.7833287715911865
Total cases = 1299, Total controls = 38012
Total cases matched is : 1179, Total control matched is : 8253
ratio: 1:7

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad     0.975      0.025
estado_inmunizacion     -2.212002  0.109481    89.051878  90.93716  86.774414

IC disntace: 0.3779707392686653


all_born                               
                      efectividad cases matched controls_matched
filtro                                                          
VRS_general  89.05 (86.77; 90.94)  1299    1179             8253